In [ ]:
import scanpy as sc
import rpy2.robjects as ro
from scipy.stats import median_abs_deviation as mad
from pathlib import Path
from src import *

%load_ext autoreload
%autoreload 2
%load_ext rpy2.ipython

sc.set_figure_params(dpi=150, facecolor='white')
sc.settings.verbosity = 3

project_root = Path.cwd().parent
data_dir = project_root / "data"
results_dir = project_root / "results"
ro.r(f'renv::load("{project_root}")')

In [ ]:
adata = sc.read_h5ad(data_dir / "processed" / "00_combined_processed.h5ad")
endothelial = adata[adata.obs["lineage"] == "endothelial"].copy()

In [ ]:
endothelial = deviant_hvg(endothelial)

In [ ]:
plot_deviance(endothelial)

In [ ]:
harmony_embed(endothelial)
sc.pl.pca_overview(endothelial, color="sample_id")

In [ ]:
cluster_subset(endothelial, n_pcs=20, resolution=0.3)

In [ ]:
sc.pl.umap(endothelial, color=["scDblFinder_score", "genotype", "pct_counts_mt", "leiden", "log1p_total_counts", "pct_counts_in_top_20_genes"], ncols=2)

In [ ]:
sc.tl.rank_genes_groups(endothelial, groupby="leiden")
sc.tl.dendrogram(endothelial, groupby="leiden", use_rep="X_pca_harmony")
sc.pl.rank_genes_groups_matrixplot(
    endothelial,
    n_genes=5, 
    groupby="leiden", 
    cmap="Reds", 
    standard_scale="var"
)

In [ ]:
endothelial_markers = {
    "gen_cap": ["Gpihbp1", "Aplnr"],
    "aerocytes": ["Car4", "Apln"],
    "proliferating cells": ["Mki67", "Top2a"],
    "arterial": ["Gja5", "Bmx"],
    "venous": ["Nr2f2", "Vwf", "Slc6a2"],
    "lymphatic": ["Prox1", "Pdpn"],
    "angiogenic": ["Angpt2"]
}

sc.pl.matrixplot(
    endothelial,
    var_names=endothelial_markers,
    groupby="leiden",
    standard_scale="var",
    dendrogram=True,
    cmap="Reds"
)

In [ ]:
endothelial_celltypes = {
    "gen_cap": ["0", "1"],
    "prolif gen_cap": ["2"],
    "arterial": ["5"],
    "venous": ["3"],
    "lymphatic": ["6"],
    "aerocytes": ["4"]
}

apply_celltypes(endothelial, endothelial_celltypes)

In [ ]:
sc.pl.umap(endothelial, color="celltype")

In [ ]:
endothelial.write_h5ad(data_dir / "processed" / "02_endothelial.h5ad")